# 10 - MobileNetV2 Partial Fine-Tuning

**RUN_E2_TRAINING=False.** Run Sections 1-3 for preparation and disposable GPU verification only. Every model starts from the hash-pinned E1 epoch-10 checkpoint; no pretrained download, previous training rerun, or test-image access.

Privately attach the existing `outputs/models/E1_mobilenetv2_frozen_seed42.pt` retaining its filename, plus the existing extracted `enge707-gpu-setup` input. No extra bundle is needed. The competition input can stay attached but is not read here.

Required working files: `/kaggle/working/enge707_train_cache/` and `/kaggle/working/training_image_cache_check.json`. Kaggle can remove these between sessions. If missing, stop: explicitly restore the verified raw-JPEG cache and report, or deliberately run notebook07's existing cache-preparation cell after its required setup. This notebook never copies images or imports another notebook. In a fresh session it can read the pinned extracted setup directly.

Separate preparation, disposable verification, and guarded training. No GPU/cross-version bitwise reproducibility or exact-resume guarantee.

## 1. Explicit definitions and trainability policy

Restore E1 with weights=None. Independently control E2 train(): selected feature modules and classifier dropout train, earlier modules stay in eval, and all BatchNorm affine parameters and running statistics stay frozen. Unfreeze only non-BatchNorm parameters in modules 16-18 plus the original classifier.

In [ ]:
RUN_E2_TRAINING = False
E2_RUN_COMPLETED = False
from pathlib import Path
import gc, hashlib, importlib, json, platform, random, statistics, struct, sys, time
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image, __version__ as pillow_version
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2

WORKING_ROOT = Path("/kaggle/working")
TRAIN_CACHE_DIR = WORKING_ROOT / "enge707_train_cache"
CACHE_REPORT = WORKING_ROOT / "training_image_cache_check.json"
SETUP_ROOT = (WORKING_ROOT / "enge707_gpu_setup" if (WORKING_ROOT / "enge707_gpu_setup").exists()
              else Path("/kaggle/input/datasets/leocaoprojects/enge707-gpu-setup"))
E1_CHECKPOINT = None  # Optional explicit path if Kaggle renames the attached file.
E1_SHA256 = "4043403fda03a02184038e817395f8001744b8542f99993383e046175b90b968"
E1_CONFIG_SHA256 = "b1be48735a9c4a942a276f2396453f26d8fc2e584ff601c1f57ce1fe2d6141cb"
TRANSFER_SHA256 = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
COUNTS = {"train": 50927, "validation": 12733}
RUN_NAME = "E2_mobilenetv2_finetuned_seed42"
DIAGNOSTIC_PATH = WORKING_ROOT / "mobilenetv2_finetuning_setup_check.json"


def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def require(condition, message):
    if not condition:
        raise ValueError(message)


def seed_all():
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


def save_json(path, value, exclusive=False):
    with Path(path).open("x" if exclusive else "w", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.write("\n")


def runtime():
    return {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__,
            "Pillow": pillow_version, "cuda_build": torch.version.cuda}

class ImageNetNormalizedDataset(Dataset):
    """Wrap verified /255 tensors; retain IDs, labels, geometry and augmentation."""
    def __init__(self, base):
        self.base = base
        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, label, image_id = self.base[index]
        if image.dtype != torch.float32 or image.shape != (3, 224, 224):
            raise ValueError("Expected reference-preprocessed float32 [3,224,224]")
        return (image - self.mean) / self.std, label, image_id

class FrozenMobileNetV2(nn.Module):
    """Official pretrained features with an eight-logit, trainable classifier."""
    def __init__(self, weights=None):
        super().__init__()
        if weights is not None and not Path("/kaggle/working").is_dir():
            raise RuntimeError("Pretrained downloads are allowed in Kaggle only")
        self.network = mobilenet_v2(weights=weights)
        self.network.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.network.last_channel, 8))
        for parameter in self.network.features.parameters():
            parameter.requires_grad_(False)
        self.train()

    def train(self, mode=True):
        super().train(mode)
        # Frozen weights alone do not freeze BatchNorm running statistics.
        self.network.features.eval()
        return self

    def forward(self, images):
        return self.network(images)

class PartiallyFineTunedMobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = mobilenet_v2(weights=None)
        self.network.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.network.last_channel, 8))
        names = list(dict(self.network.features.named_children()))
        require(names == [str(i) for i in range(19)], "Unexpected torchvision feature-module layout")
        self.selected_names = names[-3:]
        require(self.selected_names == ["16", "17", "18"]
                and [type(self.network.features[int(i)]).__name__ for i in self.selected_names]
                == ["InvertedResidual", "InvertedResidual", "Conv2dNormActivation"], "Final modules changed")
        self.requires_grad_(False)
        for name in self.selected_names:
            self.network.features[int(name)].requires_grad_(True)
        self.network.classifier.requires_grad_(True)
        for module in self.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.requires_grad_(False)
        require(sum(p.numel() for p in self.parameters()) == 2234120
                and sum(p.numel() for p in self.parameters() if p.requires_grad) == 1205128,
                "Architecture/trainable parameter count changed")
        self.train()

    def train(self, mode=True):
        # Independent from E1.train(): selected features must be allowed to train.
        super().train(mode)
        for name, module in self.network.features.named_children():
            module.train(mode and name in self.selected_names)
        for module in self.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        return self

    def forward(self, images):
        return self.network(images)  # Same eight raw logits and classifier as E1.


def parameter_groups(model):
    features = [p for p in model.network.features.parameters() if p.requires_grad]
    classifier = [p for p in model.network.classifier.parameters() if p.requires_grad]
    require(sum(p.numel() for p in features) == 1194880 and sum(p.numel() for p in classifier) == 10248,
            "Feature/classifier counts changed")
    require(not set(map(id, features)) & set(map(id, classifier)), "Optimizer groups overlap")
    return [{"params": features, "lr": 1e-5, "name": "features"},
            {"params": classifier, "lr": 1e-4, "name": "classifier"}]


def frozen_snapshot(model):
    names = {name for name, p in model.named_parameters() if not p.requires_grad}
    names.update(name for name, _ in model.named_buffers())
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items() if name in names}


def verify_frozen(model, before):
    state = model.state_dict()
    require(all(torch.equal(state[name].detach().cpu(), value) for name, value in before.items()),
            "Frozen parameters or buffers changed")
    require(all(p.grad is None for p in model.parameters() if not p.requires_grad), "Frozen gradient found")
    require(all(not module.training and all(not p.requires_grad for p in module.parameters(recurse=False))
                for module in model.modules() if isinstance(module, nn.modules.batchnorm._BatchNorm)),
            "BatchNorm policy changed")


def batch_step(model, batch, device, optimizer=None):
    images, targets, ids = batch
    require(images.dtype == torch.float32 and targets.dtype == torch.int64, "Invalid batch dtype")
    require(bool(torch.isfinite(images).all()) and bool(((targets >= 0) & (targets < 8)).all()), "Invalid inputs/labels")
    images, targets = images.to(device), targets.to(device)
    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
    with (torch.enable_grad() if optimizer is not None else torch.inference_mode()), torch.autocast(device_type=device.type, enabled=False):
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, targets)  # Unweighted raw-logit CE.
        require(logits.shape == (len(ids), 8) and bool(torch.isfinite(logits).all() & torch.isfinite(loss)),
                "Invalid logits or loss")
        if optimizer is not None:
            loss.backward()
            for group in optimizer.param_groups:
                gradients = [p.grad for p in group["params"]]
                require(all(g is not None and bool(torch.isfinite(g).all()) for g in gradients),
                        "Missing/nonfinite trainable gradients")
                require(any(bool((g != 0).any()) for g in gradients), "All gradients zero in an optimizer group")
            optimizer.step()
    return {"loss_sum": float(loss.detach()) * len(ids),
            "correct": int((logits.detach().argmax(1) == targets).sum()), "samples": len(ids)}


def run_epoch(model, loader, device, optimizer=None):
    model.train(optimizer is not None)
    loss_sum = correct = samples = 0
    seen = []
    expected = loader.dataset.base.rows["image_id"].tolist()
    for batch in loader:
        result = batch_step(model, batch, device, optimizer)
        loss_sum += result["loss_sum"]
        correct += result["correct"]
        samples += result["samples"]
        seen.extend(batch[2])
    require(samples == len(expected) and samples > 0 and len(set(seen)) == samples
            and set(seen) == set(expected), "Incomplete/repeated epoch IDs")
    require(optimizer is not None or seen == expected, "Validation order changed")
    return {"loss": loss_sum / samples, "accuracy": correct / samples, "samples": samples}


## 2. Preparation only

Fail before model construction or image decoding if the checkpoint, setup, or verified cache/report is missing. Verify the pinned manifest and all six payloads before importing the original Dataset. Require the same 12 strict preprocessing fingerprints and unchanged split/mapping. Filename enumeration plus the prior full cache hash report verify cache coverage; this does not repeat the whole-cache byte audit.

In [ ]:
def restore_e1(path, fine_tune=True):
    require(digest(path) == E1_SHA256, "E1 checkpoint SHA256 mismatch")
    checkpoint = torch.load(path, weights_only=True, map_location="cpu")
    canonical = json.dumps(checkpoint["configuration"], sort_keys=True, separators=(",", ":")).encode()
    require(hashlib.sha256(canonical).hexdigest() == E1_CONFIG_SHA256, "E1 configuration changed")
    require(checkpoint["best_epoch"] == 10 and checkpoint["best_validation_loss"] == 0.1460895589143588,
            "Wrong E1 epoch or best loss")
    model = PartiallyFineTunedMobileNetV2() if fine_tune else FrozenMobileNetV2(weights=None)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model, checkpoint


def make_loaders(prepared):
    seed_all()
    bases = {name: FacialFeatureDataset(prepared["setup"], name, training_image_directory=TRAIN_CACHE_DIR)
             for name in COUNTS}
    for name, base in bases.items():
        require(len(base) == COUNTS[name] and base.rows.equals(prepared["rows"][name][["image_id", "class", "image_path"]]),
                "Dataset membership/order changed")
    require(bases["validation"].augmentation is None
            and repr(bases["train"].augmentation) == prepared["e1_config"]["training_augmentation"],
            "Original augmentation changed")
    datasets = {name: ImageNetNormalizedDataset(base) for name, base in bases.items()}
    return {"train": DataLoader(datasets["train"], batch_size=32, num_workers=0, drop_last=False,
                               shuffle=True, generator=torch.Generator().manual_seed(42)),
            "validation": DataLoader(datasets["validation"], batch_size=32, num_workers=0, drop_last=False, shuffle=False)}


def verify_inputs(prepared):
    for name, path in prepared["input_paths"].items():
        require(digest(path) == prepared["input_sha256"][name], "Input changed: " + name)


def prepare_e2():
    require(WORKING_ROOT.is_dir(), "Kaggle /kaggle/working required; do not run preparation locally")
    require(TRAIN_CACHE_DIR.is_dir() and not TRAIN_CACHE_DIR.is_symlink() and CACHE_REPORT.is_file(),
            "Missing verified cache/report: restore /kaggle/working/enge707_train_cache and "
            "training_image_cache_check.json, or explicitly run notebook07's cache-preparation cell "
            "after its required verified setup. No automatic dataset copy is performed.")
    require(SETUP_ROOT.is_dir() and not SETUP_ROOT.is_symlink(),
            "Attach the existing extracted enge707-gpu-setup input or restore the verified working setup.")
    if E1_CHECKPOINT is not None:
        candidates = [Path(E1_CHECKPOINT)]
    else:
        candidates = list(Path("/kaggle/input").rglob("E1_mobilenetv2_frozen_seed42.pt"))
        hosted = WORKING_ROOT / "E1_mobilenetv2_frozen_seed42/best_checkpoint.pt"
        if hosted.is_file():
            candidates.append(hosted)
    require(len(candidates) == 1 and candidates[0].is_file() and not candidates[0].is_symlink(),
            "Attach exactly one E1_mobilenetv2_frozen_seed42.pt privately, or set E1_CHECKPOINT explicitly.")
    checkpoint_path = candidates[0].resolve()
    require(digest(checkpoint_path) == E1_SHA256, "Transferred E1 checkpoint differs from local original")
    manifest_path = SETUP_ROOT / "transfer_manifest.json"
    require(digest(manifest_path) == TRANSFER_SHA256, "Pinned transfer manifest mismatch")
    transfer = json.loads(manifest_path.read_text())
    expected_payload = {"src/__init__.py", "src/data_pipeline.py", "src/models.py", "data/processed/split_manifest.csv",
                        "data/processed/class_mapping.json", "preprocessing_fingerprints.json"}
    require(set(transfer["sha256"]) == expected_payload, "Unexpected transfer payload")
    input_paths = {name: SETUP_ROOT / name for name in expected_payload}
    for name, path in input_paths.items():
        require(not path.is_symlink() and path.resolve().is_relative_to(SETUP_ROOT.resolve())
                and not any(parent.is_symlink() for parent in path.parents), "Unsafe payload path")
        require(digest(path) == transfer["sha256"][name], "Payload checksum mismatch: " + name)
    input_paths.update(transfer_manifest=manifest_path, e1_checkpoint=checkpoint_path, cache_report=CACHE_REPORT)
    split = pd.read_csv(input_paths["data/processed/split_manifest.csv"], dtype=str, keep_default_na=False, na_filter=False)
    mapping = json.loads(input_paths["data/processed/class_mapping.json"].read_text())
    names = sorted(mapping["class_to_index"])
    require(len(names) == 8 and mapping == transfer["classes"] and mapping["class_to_index"] == dict(zip(names, range(8)))
            and all(mapping["index_to_class"][str(i)] == name for i, name in enumerate(names)), "Mapping changed")
    require(len(split) == 63660 and split["image_id"].is_unique and split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            and split["split"].value_counts().to_dict() == COUNTS
            and split["image_path"].equals("data/raw/train/train/" + split["image_id"] + ".jpg"), "Split contract changed")
    rows = {name: split.loc[split["split"].eq(name)].reset_index(drop=True) for name in COUNTS}
    overlaps = {key: len(set(rows["train"][key]) & set(rows["validation"][key]))
                for key in ["image_id", "byte_sha256", "pixel_sha256"]}
    require(not any(overlaps.values()) and all(set(frame["class"]) == set(names) for frame in rows.values()),
            "Split overlap or class coverage failure")
    cache_report = json.loads(CACHE_REPORT.read_text())
    require(cache_report.get("status") == "passed" and cache_report.get("verified_files") == 63660
            and cache_report.get("hash_checks") == 63660 and not cache_report.get("failures")
            and not cache_report.get("hash_mismatches") and cache_report.get("checks")
            and all(cache_report["checks"].values())
            and Path(cache_report["cache_directory"]).resolve() == TRAIN_CACHE_DIR.resolve(), "Incomplete cache report")
    files = list(TRAIN_CACHE_DIR.iterdir())
    require(len(files) == 63660 and {path.name for path in files} == set(split["image_id"] + ".jpg")
            and all(path.is_file() and not path.is_symlink() and path.resolve().parent == TRAIN_CACHE_DIR.resolve()
                    for path in files), "Missing/extra/unsafe cache files")
    # Import original source only after all payload bytes pass verification.
    for name, module in list(sys.modules.items()):
        if (name == "src" or name.startswith("src.")) and getattr(module, "__file__", None):
            require(Path(module.__file__).resolve().is_relative_to(SETUP_ROOT.resolve()),
                    "Different src code already imported; use a clean verified session")
    if str(SETUP_ROOT.resolve()) not in sys.path:
        sys.path.insert(0, str(SETUP_ROOT.resolve()))
    global FacialFeatureDataset
    pipeline = importlib.import_module("src.data_pipeline")
    FacialFeatureDataset = pipeline.FacialFeatureDataset
    fingerprints = json.loads(input_paths["preprocessing_fingerprints.json"].read_text())["examples"]
    require(len(fingerprints) == len({item["image_id"] for item in fingerprints}) == 12, "Fingerprint set changed")
    for item in fingerprints:
        row = split.loc[split["image_id"].eq(item["image_id"])].iloc[0]
        require(row["split"] == "train" and row["class"] == item["class"], "Fingerprint membership changed")
        path = TRAIN_CACHE_DIR / (item["image_id"] + ".jpg")
        require(digest(path) == item["original_byte_sha256"] == row["byte_sha256"], "Fingerprint JPEG changed")
        with Image.open(path) as image:
            image.load()
            tensor = pipeline.reference_preprocess(image)
        header = b"ENGE707-reference-f32-v1\0" + struct.pack(">3Q", *tensor.shape)
        actual = hashlib.sha256(header + np.asarray(tensor.numpy(), dtype="<f4").tobytes(order="C")).hexdigest()
        require(list(tensor.shape) == [3, 224, 224] and tensor.dtype == torch.float32
                and actual == item["tensor_sha256"], "Preprocessing fingerprint mismatch; do not relax across versions")
    seed_all()
    model, checkpoint = restore_e1(checkpoint_path)
    config = checkpoint["configuration"]
    require(checkpoint["class_mapping"] == mapping == config["class_mapping"]
            and checkpoint["split_checksum"] == digest(input_paths["data/processed/split_manifest.csv"])
            and config["counts"] == COUNTS
            and all(config["input_sha256"][name] == transfer["sha256"][name] for name in expected_payload),
            "E1 metadata differs from verified setup")
    print("Selected modules:", ["network.features." + name for name in model.selected_names])
    print("Total/trainable parameters:", sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    print("LRs: features=1e-5; classifier=1e-4; all BatchNorm parameters/buffers frozen")
    prepared = {"setup": SETUP_ROOT, "checkpoint": checkpoint_path, "mapping": mapping, "rows": rows,
                "e1_config": config, "overlap_counts": overlaps, "input_paths": input_paths,
                "input_sha256": {name: digest(path) for name, path in input_paths.items()}}
    parameter_groups(model)
    del model, checkpoint
    loaders = make_loaders(prepared)
    raw, label, image_id = loaders["validation"].dataset.base[0]
    normalised, repeated_label, repeated_id = loaders["validation"].dataset[0]
    dataset = loaders["validation"].dataset
    require(torch.equal(normalised, (raw - dataset.mean) / dataset.std)
            and image_id == repeated_id and label == repeated_label, "Normalisation changed")
    verify_inputs(prepared)
    print("Preparation passed. Next: Section 3 only. Full E2 training is disabled.")
    return prepared

preparation = prepare_e2()


## 3. Disposable GPU verification only

Compare two fixed, unaugmented validation batches before updates against an independent original E1 model. Require eight finite logits matching with `atol=1e-6, rtol=1e-5`; save actual maximum differences. Then run exactly two warm-up training batches, three timed training batches and two timed validation batches on the disposable E2 model. Timings include batch loading, transfer, forward/loss and training backward/Adam, with CUDA synchronisation at boundaries.

Check a 120-second budget between batches; incomplete counts fail the diagnostic and block training. One batch may exceed the budget. Verify updates in each selected feature module and classifier, with every frozen parameter and buffer unchanged. Record GPU peak allocated/reserved memory. An existing diagnostic is protected. Discard diagnostic weights and optimizer state. Full-epoch projections are short-sample estimates, not measured epochs.

In [ ]:
def verify_e2_gpu(prepared):
    require(not DIAGNOSTIC_PATH.exists(), "Diagnostic already exists; preserve it before a deliberate rerun")
    result = {"status": "running", "diagnostic_only": True, "checks": {}, "failures": [],
              "runtime": runtime(), "input_sha256": prepared["input_sha256"],
              "configuration": {"seed": 42, "batch_size": 32, "num_workers": 0, "drop_last": False,
                                "device": "cuda:0", "dtype": "float32", "amp": False,
                                "feature_lr": 1e-5, "classifier_lr": 1e-4},
              "batch_seconds": {"warmup": [], "training": [], "validation": []},
              "comparison_atol": 1e-6, "comparison_rtol": 1e-5, "comparison_max_abs_differences": [],
              "limitations": ["Rough epoch projection is a short-sample estimate, not a measured epoch.",
                              "Timing includes loading, transfer, verification and cache effects.",
                              "No GPU/cross-version bitwise reproducibility or exact-resume guarantee."]}
    model = baseline = optimizer = loaders = batch = images = reference = actual = None
    train_iterator = val_iterator = iterator = None
    try:
        verify_inputs(prepared)
        require(torch.cuda.is_available(), "CUDA unavailable; enable one Kaggle GPU and use cuda:0")
        device = torch.device("cuda:0")
        result["device"] = {"selected": str(device), "name": torch.cuda.get_device_name(0),
                            "free_bytes": torch.cuda.mem_get_info(0)[0], "cpu_threads": torch.get_num_threads()}
        seed_all()
        model, _ = restore_e1(prepared["checkpoint"])
        baseline, _ = restore_e1(prepared["checkpoint"], fine_tune=False)
        model, baseline = model.to(device=device, dtype=torch.float32), baseline.to(device=device, dtype=torch.float32)
        model.eval(); baseline.eval()
        loaders = make_loaders(prepared)
        iterator = iter(loaders["validation"])
        with torch.inference_mode():
            for number in range(2):
                batch = next(iterator)
                require(list(batch[2]) == prepared["rows"]["validation"]["image_id"].iloc[number*32:(number+1)*32].tolist(),
                        "Fixed comparison validation order changed")
                images = batch[0].to(device)
                reference, actual = baseline(images), model(images)
                require(actual.shape == (32, 8) and bool(torch.isfinite(actual).all() & torch.isfinite(reference).all()),
                        "Invalid comparison logits")
                result["comparison_max_abs_differences"].append(float((actual - reference).abs().max()))
                require(torch.allclose(actual, reference, atol=1e-6, rtol=1e-5), "Restored E1 equivalence failed")
        result["checks"]["two_fixed_validation_batches_match_original_e1"] = True
        baseline = None
        frozen = frozen_snapshot(model)
        before = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
        optimizer = torch.optim.Adam(parameter_groups(model))
        model.train()
        require(model.network.classifier[0].training
                and all(model.network.features[int(name)].training for name in model.selected_names),
                "Selected features/classifier dropout did not enter train mode")
        verify_frozen(model, frozen)
        result["selected_modules"] = ["network.features." + name for name in model.selected_names]
        result["parameters"] = {"total": sum(p.numel() for p in model.parameters()),
                                "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
        loaders = make_loaders(prepared)
        labels_by_id = {name: rows.set_index("image_id")["class"].map(prepared["mapping"]["class_to_index"])
                        for name, rows in prepared["rows"].items()}
        train_iterator, val_iterator = iter(loaders["train"]), iter(loaders["validation"])
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        started = time.perf_counter()
        for phase, count in [("warmup", 2), ("training", 3), ("validation", 2)]:
            model.train(phase != "validation")
            for number in range(count):
                require(time.perf_counter() - started < 120, "120s diagnostic budget exhausted; no training enabled")
                torch.cuda.synchronize(device)
                tick = time.perf_counter()
                batch = next(val_iterator if phase == "validation" else train_iterator)
                partition = "validation" if phase == "validation" else "train"
                require(batch[1].tolist() == labels_by_id[partition].loc[list(batch[2])].tolist(), "Batch labels changed")
                batch_step(model, batch, device, None if phase == "validation" else optimizer)
                torch.cuda.synchronize(device)
                result["batch_seconds"][phase].append(time.perf_counter() - tick)
                print(phase, number + 1, f"{result['batch_seconds'][phase][-1]:.3f}s", flush=True)
        verify_frozen(model, frozen)
        changed = {name for name, p in model.named_parameters() if name in before
                   and not torch.equal(p.detach().cpu(), before[name])}
        require(any(name.startswith("network.classifier.") for name in changed), "Classifier did not update")
        for name in model.selected_names:
            require(any(key.startswith("network.features." + name + ".") for key in changed), "Selected module did not update: " + name)
        verify_inputs(prepared)
        result["checks"].update(finite_loss_and_gradients=True, selected_feature_layers_updated=True,
            classifier_updated=True, frozen_parameters_and_all_buffers_unchanged=True,
            train_modes_and_bn_policy=True, original_inputs_unchanged=True)
        result["completed_batches"] = {phase: len(times) for phase, times in result["batch_seconds"].items()}
        result["mean_batch_seconds"] = {phase: statistics.mean(times) for phase, times in result["batch_seconds"].items()}
        result["loader_batch_counts"] = {name: len(loader) for name, loader in loaders.items()}
        result["rough_epoch_estimate_seconds"] = (result["mean_batch_seconds"]["training"] * len(loaders["train"])
                                                  + result["mean_batch_seconds"]["validation"] * len(loaders["validation"]))
        result["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated(device)
        result["peak_gpu_reserved_bytes"] = torch.cuda.max_memory_reserved(device)
        result["timing_phase_seconds"] = time.perf_counter() - started
        result["status"] = "passed"
    except Exception as error:
        result["status"] = "failed"
        result["failures"].append(f"{type(error).__name__}: {error}")
        raise
    finally:
        model = baseline = optimizer = loaders = batch = images = reference = actual = None
        train_iterator = val_iterator = iterator = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        result["completed_batches"] = {phase: len(times) for phase, times in result["batch_seconds"].items()}
        result["mean_batch_seconds"] = {phase: statistics.mean(times) if times else None
                                         for phase, times in result["batch_seconds"].items()}
        result["diagnostic_model_and_optimizer_discarded"] = True
        save_json(DIAGNOSTIC_PATH, result, exclusive=True)
        print("Diagnostic:", result["status"], DIAGNOSTIC_PATH)
    print("Rough epoch estimate (NOT measured):", result["rough_epoch_estimate_seconds"] / 60, "minutes")
    print("Full E2 training remains disabled.")
    return result

finetuning_setup_check = verify_e2_gpu(preparation)


## 4. Guarded E2 training - disabled

Do not enable yet. Later, require successful live/on-disk GPU diagnostics, unchanged inputs and the same runtime. Reserve the new E2 directory exclusively; never resume or overwrite an existing run. Reset seeds and restore a fresh model from the original E1 checkpoint, with fresh loaders and Adam state.

Maximum 8 epochs, early stopping after 3 consecutive epochs without strictly lower validation loss, ReduceLROnPlateau factor 0.5/patience 2/absolute threshold 0. Keep sample-weighted loss/accuracy and full epoch coverage, including partial batches. Log both learning rates before and after scheduling. Save configuration, history after each epoch, best checkpoint by validation loss and completion status. No E0/E1 files are written.

In [ ]:
RUN_E2_TRAINING = False
E2_RUN_COMPLETED = False


def train_e2(prepared, diagnostic):
    require(DIAGNOSTIC_PATH.is_file(), "Run GPU verification first")
    saved = json.loads(DIAGNOSTIC_PATH.read_text())
    require(saved == diagnostic and saved["status"] == "passed" and not saved["failures"]
            and saved["checks"] and all(saved["checks"].values())
            and saved["completed_batches"] == {"warmup": 2, "training": 3, "validation": 2}
            and saved["diagnostic_model_and_optimizer_discarded"]
            and saved["input_sha256"] == prepared["input_sha256"]
            and saved["runtime"] == runtime(), "Successful unchanged hosted diagnostic required")
    require(torch.cuda.is_available() and torch.cuda.get_device_name(0) == saved["device"]["name"], "GPU changed")
    verify_inputs(prepared)
    run = WORKING_ROOT / RUN_NAME
    require(not run.exists(), "E2 output directory exists; no overwrite or automatic resume")
    device = torch.device("cuda:0")
    seed_all()
    loaders = make_loaders(prepared)
    model, checkpoint = restore_e1(prepared["checkpoint"])
    model = model.to(device=device, dtype=torch.float32)
    frozen = frozen_snapshot(model)
    optimizer = torch.optim.Adam(parameter_groups(model))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
        patience=2, threshold=0, threshold_mode="abs")
    configuration = {"run_name": RUN_NAME, "seed": 42, "max_epochs": 8, "early_stopping_patience": 3,
        "optimizer": "Adam", "initial_learning_rates": {"features": 1e-5, "classifier": 1e-4},
        "loss": "unweighted CrossEntropyLoss on raw logits", "batch_size": 32, "num_workers": 0,
        "drop_last": False, "train_shuffle": True, "validation_shuffle": False,
        "device": "cuda:0", "dtype": "float32", "amp": False, "multi_gpu": False,
        "scheduler": {"name": "ReduceLROnPlateau", "factor": 0.5, "patience": 2, "threshold": 0, "threshold_mode": "abs"},
        "selected_modules": saved["selected_modules"], "parameters": saved["parameters"],
        "trainability_policy": "Final three feature modules' non-BatchNorm parameters and classifier train; all BN parameters and buffers frozen",
        "architecture": prepared["e1_config"]["architecture"], "class_mapping": prepared["mapping"], "counts": COUNTS,
        "preprocessing": prepared["e1_config"]["preprocessing"],
        "training_augmentation": prepared["e1_config"]["training_augmentation"], "validation_augmentation": None,
        "training_image_directory": str(TRAIN_CACHE_DIR), "input_sha256": prepared["input_sha256"],
        "split_sha256": prepared["input_sha256"]["data/processed/split_manifest.csv"],
        "e1_checkpoint": {"path": str(prepared["checkpoint"]), "sha256": E1_SHA256, "best_epoch": 10,
                          "best_validation_loss": checkpoint["best_validation_loss"], "original_weights": prepared["e1_config"]["weights"]},
        "diagnostic_sha256": digest(DIAGNOSTIC_PATH), "runtime": runtime(), "gpu": saved["device"],
        "best_checkpoint_rule": "lowest validation loss", "exact_resume_supported": False}
    run.mkdir(exist_ok=False)
    save_json(run / "configuration.json", configuration, exclusive=True)
    completion = {"status": "running", "completed_epochs": 0, "failures": []}
    save_json(run / "completion_status.json", completion, exclusive=True)
    history = []
    best_loss, best_epoch, stale = float("inf"), None, 0
    torch.cuda.synchronize(device)
    started = time.perf_counter()
    try:
        for epoch in range(1, 9):
            torch.cuda.synchronize(device)
            tick = time.perf_counter()
            rates = {group["name"]: float(group["lr"]) for group in optimizer.param_groups}
            training = run_epoch(model, loaders["train"], device, optimizer)
            validation = run_epoch(model, loaders["validation"], device)
            require(training["samples"] == 50927 and validation["samples"] == 12733, "Wrong epoch sample totals")
            verify_frozen(model, frozen)
            if validation["loss"] < best_loss:
                best_loss, best_epoch, stale = validation["loss"], epoch, 0
                torch.save({"model_state_dict": model.state_dict(), "class_mapping": prepared["mapping"],
                            "configuration": configuration, "split_checksum": configuration["split_sha256"],
                            "best_epoch": best_epoch, "best_validation_loss": best_loss}, run / "best_checkpoint.pt")
            else:
                stale += 1
            scheduler.step(validation["loss"])
            torch.cuda.synchronize(device)
            row = {"epoch": epoch, "training_loss": training["loss"], "training_accuracy": training["accuracy"],
                   "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"],
                   "training_samples": training["samples"], "validation_samples": validation["samples"],
                   "feature_learning_rate": rates["features"], "classifier_learning_rate": rates["classifier"],
                   "next_feature_learning_rate": optimizer.param_groups[0]["lr"],
                   "next_classifier_learning_rate": optimizer.param_groups[1]["lr"],
                   "epoch_elapsed_seconds": time.perf_counter() - tick, "elapsed_seconds": time.perf_counter() - started}
            history.append(row)
            pd.DataFrame(history).to_csv(run / "history.csv", index=False)
            print(json.dumps(row), flush=True)
            completion.update(completed_epochs=epoch, best_epoch=best_epoch, best_validation_loss=best_loss,
                              elapsed_seconds=row["elapsed_seconds"])
            save_json(run / "completion_status.json", completion)
            if stale >= 3:
                break
        verify_frozen(model, frozen)
        verify_inputs(prepared)
        completion.update(status="completed", frozen_parameters_and_bn_buffers_unchanged=True,
                          history_sha256=digest(run / "history.csv"),
                          stop_reason="early stopping" if stale >= 3 else "maximum epochs")
    except Exception as error:
        completion.update(status="failed", failures=[f"{type(error).__name__}: {error}"])
        raise
    finally:
        save_json(run / "completion_status.json", completion)
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()
    print("E2 completed:", completion)
    return completion

if RUN_E2_TRAINING:
    e2_completion = train_e2(preparation, finetuning_setup_check)
    E2_RUN_COMPLETED = True
else:
    print("RUN_E2_TRAINING=False: full training and experiment outputs skipped.")


## 5. Guarded learning curves

Plot only actual, successfully completed E2 history, never diagnostic results. Protect existing curves.

In [ ]:
if RUN_E2_TRAINING:
    require(E2_RUN_COMPLETED and e2_completion["status"] == "completed", "Completed E2 run required")
    run = WORKING_ROOT / RUN_NAME
    completion = json.loads((run / "completion_status.json").read_text())
    require(completion == e2_completion and digest(run / "history.csv") == completion["history_sha256"],
            "Completed history changed")
    history = pd.read_csv(run / "history.csv")
    require(history["epoch"].tolist() == list(range(1, completion["completed_epochs"] + 1))
            and history["training_samples"].eq(50927).all() and history["validation_samples"].eq(12733).all(),
            "Invalid E2 history")
    figure_path = run / "loss_accuracy_curves.png"
    require(not figure_path.exists(), "Curves already exist; no overwrite")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(history["epoch"], history["training_" + metric], label="Training")
        ax.plot(history["epoch"], history["validation_" + metric], label="Validation")
        ax.set(xlabel="Epoch", ylabel=metric.capitalize(), title="E2 " + metric)
        ax.legend()
    with figure_path.open("xb") as stream:
        fig.savefig(stream, format="png", dpi=300)
    plt.show()
    plt.close(fig)
    print("Saved:", figure_path)
else:
    print("RUN_E2_TRAINING=False: learning curves skipped.")


## 6. E2 evaluation only - paste the final code cell into the current Kaggle notebook

Run **only the following new cell** after the completed E2 run. Do not rerun preparation, GPU diagnostics, copying or training. All training flags are reset to False. It evaluates the complete saved validation partition using the original deterministic geometry, ImageNet normalisation and sequential batch32 loader (workers0, final partial batch retained).

Required existing definitions: `PartiallyFineTunedMobileNetV2` (its constructor uses `mobilenet_v2(weights=None)`), `ImageNetNormalizedDataset`, `SETUP_ROOT`, `TRAIN_CACHE_DIR`, the model's existing `require` assertion helper, and the already verified/imported `src.data_pipeline.FacialFeatureDataset`. Missing state causes an explicit stop; the cell never recreates earlier state or downloads weights. The original E1 checkpoint and live diagnostic variables are not needed for E2 inference.

Required files: the completed E2 run's `best_checkpoint.pt`, `configuration.json`, `history.csv`, `completion_status.json`; the pinned setup's six payload files and `transfer_manifest.json`; and `/kaggle/working/training_image_cache_check.json` with the existing raw-JPEG working cache. Verify source, split, mapping, completed-history and checkpoint hashes before evaluation. An existing `evaluation/` directory is left untouched; a new directory is reserved exclusively.

Compare restored accuracy with 0.9862561847168774 using absolute tolerance 1e-12, and sample-weighted cross-entropy with 0.040039216507934836 using absolute tolerance 1e-6 (relative tolerance 0). Record actual differences and retain actual results with a discrepancy status/warning if outside tolerance. No results are changed to force agreement. Confidence means maximum softmax score, not calibrated certainty. Rows in both confusion matrices are true classes; columns are predicted classes in saved index order.


In [ ]:
# 6. E2 EVALUATION ONLY - paste and run only this cell in the current session.
RUN_E2_TRAINING = False
RUN_E1_TRAINING = False
RUN_FULL_TRAINING = False
try:
    import hashlib
    import json
    import math
    import platform
    import sys
    from pathlib import Path
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    import torchvision
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision.models import mobilenet_v2
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
except ImportError as error:
    raise RuntimeError("Required evaluation dependency is unavailable; nothing installed") from error


def _e2_validation_statistics(true, predicted, loss_sum, names):
    """Use explicit class order; undefined precision/F1 is reported as zero."""
    precision, recall, f1, support = precision_recall_fscore_support(
        true, predicted, labels=list(range(8)), zero_division=0)
    matrix = confusion_matrix(true, predicted, labels=list(range(8)))
    per_class = pd.DataFrame({"class_index": list(range(8)), "class": names,
        "precision": precision, "recall": recall, "f1": f1, "support": support})
    scores = {"samples": len(true), "cross_entropy": loss_sum / len(true),
        "accuracy": float(accuracy_score(true, predicted)), "macro_f1": float(f1.mean()),
        "weighted_f1": float(np.average(f1, weights=support))}
    return scores, per_class, matrix


def _evaluate_saved_e2_validation():
    required = ["PartiallyFineTunedMobileNetV2", "ImageNetNormalizedDataset", "SETUP_ROOT", "TRAIN_CACHE_DIR", "require"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Missing existing E2 session definitions: " + ", ".join(missing)
                           + ". Use the session that completed E2; this cell does not rerun setup or training.")
    run = Path("/kaggle/working/E2_mobilenetv2_finetuned_seed42")
    destination = run / "evaluation"
    if destination.exists() or destination.is_symlink():
        raise FileExistsError(f"Existing evaluation directory left unchanged: {destination}")
    setup, cache = Path(SETUP_ROOT).resolve(), Path(TRAIN_CACHE_DIR).resolve()
    paths = {"checkpoint": run / "best_checkpoint.pt", "configuration": run / "configuration.json",
             "history": run / "history.csv", "completion_status": run / "completion_status.json",
             "transfer_manifest": setup / "transfer_manifest.json",
             "cache_report": run.parent / "training_image_cache_check.json"}
    for path in paths.values():
        if not path.is_file() or path.is_symlink():
            raise FileNotFoundError(f"Required existing file unavailable/unsafe: {path}; nothing recreated")
    if not cache.is_dir() or cache != (run.parent / "enge707_train_cache").resolve():
        raise FileNotFoundError("Required existing /kaggle/working/enge707_train_cache is missing; no images copied")

    def digest(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    def require(condition, message):
        if not condition:
            raise ValueError(message)

    # Pins were recorded from the preserved completed E2 exports, not regenerated.
    pinned = {"checkpoint": "89718e38fc47bff1fa31ed923ebab9a5a2aefe7fdea27fe73334cb366443d688",
              "configuration": "98c31a5c619f3c9a44731cf96bb43ea75955c071f48345b54a862658cbaf2c90",
              "history": "ea2f434868afc1edc86205f41065099635924bb06cb8c52b5b26833eeeca84b4",
              "transfer_manifest": "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"}
    for name, expected_hash in pinned.items():
        require(digest(paths[name]) == expected_hash, "Pinned E2/transfer checksum changed: " + name)
    transfer = json.loads(paths["transfer_manifest"].read_text())
    expected_payload = {"src/__init__.py", "src/data_pipeline.py", "src/models.py",
                        "data/processed/split_manifest.csv", "data/processed/class_mapping.json",
                        "preprocessing_fingerprints.json"}
    require(set(transfer["sha256"]) == expected_payload, "Unexpected setup payload")
    for name in expected_payload:
        path = setup / name
        require(path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(setup),
                "Required verified setup file missing/unsafe: " + name)
        require(digest(path) == transfer["sha256"][name], "Setup payload checksum changed: " + name)
        paths[name] = path
    before = {name: digest(path) for name, path in paths.items()}
    config = json.loads(paths["configuration"].read_text())
    completion = json.loads(paths["completion_status"].read_text())
    history = pd.read_csv(paths["history"], float_precision="round_trip")
    checkpoint = torch.load(paths["checkpoint"], weights_only=True, map_location="cpu")
    require(config["run_name"] == run.name and config["seed"] == 42 and config["max_epochs"] == 8
            and checkpoint["configuration"] == config and checkpoint["best_epoch"] == 8
            and config["selected_modules"] == ["network.features.16", "network.features.17", "network.features.18"]
            and config["parameters"] == {"total": 2234120, "trainable": 1205128}, "E2 checkpoint/configuration changed")
    require(history["epoch"].tolist() == list(range(1, 9))
            and history["training_samples"].eq(50927).all() and history["validation_samples"].eq(12733).all()
            and int(history.loc[history["validation_loss"].idxmin(), "epoch"]) == 8,
            "Completed E2 history does not describe eight epochs with best epoch 8")
    epoch8 = history.iloc[-1]
    expected_accuracy, expected_loss = 0.9862561847168774, 0.040039216507934836
    require(float(epoch8["validation_accuracy"]) == expected_accuracy
            and float(epoch8["validation_loss"]) == checkpoint["best_validation_loss"] == expected_loss,
            "E2 epoch-8 reference metrics changed")
    require(completion["status"] == "completed" and completion["completed_epochs"] == 8
            and completion["best_epoch"] == 8 and completion["best_validation_loss"] == expected_loss
            and not completion["failures"] and completion["frozen_parameters_and_bn_buffers_unchanged"] is True
            and completion["history_sha256"] == before["history"], "Successful E2 completion evidence changed")
    # E1 provenance is recorded in the complete E2 checkpoint; E1 weights need not be loaded again.
    require(config["e1_checkpoint"]["sha256"] == config["input_sha256"]["e1_checkpoint"]
            == "4043403fda03a02184038e817395f8001744b8542f99993383e046175b90b968",
            "E1 origin of E2 changed")
    require(all(before[name] == expected_hash for name, expected_hash in config["input_sha256"].items()
                if name != "e1_checkpoint"), "Recorded source/split/mapping/cache hashes changed")
    module = sys.modules.get("src.data_pipeline")
    if module is None or not getattr(module, "__file__", None) or Path(module.__file__).resolve() != setup / "src/data_pipeline.py":
        raise RuntimeError("Existing verified src.data_pipeline.FacialFeatureDataset is required; no source imported/rebuilt")
    FacialFeatureDataset = module.FacialFeatureDataset
    mapping = json.loads(paths["data/processed/class_mapping.json"].read_text())
    names = [mapping["index_to_class"][str(i)] for i in range(8)]
    require(names == sorted(names) and len(set(names)) == 8
            and mapping["class_to_index"] == dict(zip(names, range(8)))
            and mapping == config["class_mapping"] == checkpoint["class_mapping"] == transfer["classes"],
            "Exact saved class mapping changed")
    require(before["data/processed/split_manifest.csv"] == config["split_sha256"] == checkpoint["split_checksum"],
            "Saved split differs from checkpoint/configuration")
    cache_report = json.loads(paths["cache_report"].read_text())
    require(cache_report.get("status") == "passed" and cache_report.get("verified_files") == 63660
            and not cache_report.get("failures") and not cache_report.get("hash_mismatches")
            and cache_report.get("checks") and all(cache_report["checks"].values())
            and Path(cache_report["cache_directory"]).resolve() == cache, "Verified working-image cache report unavailable")
    split = pd.read_csv(paths["data/processed/split_manifest.csv"], dtype=str, keep_default_na=False, na_filter=False)
    require(len(split) == 63660 and split["image_id"].is_unique
            and split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            and split["split"].value_counts().to_dict() == {"train": 50927, "validation": 12733},
            "Saved split coverage/IDs changed")
    expected = split.loc[split["split"].eq("validation")].reset_index(drop=True)
    train_rows = split.loc[split["split"].eq("train")]
    require(not any(set(train_rows[key]) & set(expected[key]) for key in ["image_id", "byte_sha256", "pixel_sha256"]),
            "Cross-partition ID/hash overlap")
    base = FacialFeatureDataset(setup, "validation", training_image_directory=cache)
    require(len(base) == 12733 and base.augmentation is None
            and base.class_to_index == mapping["class_to_index"]
            and base.rows.equals(expected[["image_id", "class", "image_path"]]), "Validation membership/labels/order changed")
    dataset = ImageNetNormalizedDataset(base)
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]
    require(torch.equal(dataset.mean, mean) and torch.equal(dataset.std, std)
            and config["preprocessing"]["mean"] == [0.485, 0.456, 0.406]
            and config["preprocessing"]["std"] == [0.229, 0.224, 0.225]
            and config["preprocessing"]["normalisation_applications"] == 1, "E1/E2 normalisation changed")
    raw, label, first_id = base[0]
    first, label1, id1 = dataset[0]
    repeated, label2, id2 = dataset[0]
    require(torch.equal(first, (raw - mean) / std) and torch.equal(first, repeated)
            and label == label1 == label2 and first_id == id1 == id2, "Preprocessing is not deterministic/normalised once")
    expected_ids = expected["image_id"].tolist()
    expected_targets = expected["class"].map(mapping["class_to_index"]).tolist()
    expected_support = [int(expected["class"].eq(name).sum()) for name in names]
    require(all(expected_support), "Expected all eight validation classes")
    if not torch.cuda.is_available():
        raise RuntimeError("Existing cuda:0 runtime required; no local full evaluation is supported")
    device = torch.device("cuda:0")
    # This E2 constructor uses mobilenet_v2(weights=None); the complete state restores all features.
    model = PartiallyFineTunedMobileNetV2().to(dtype=torch.float32)
    parameters = {"total": sum(p.numel() for p in model.parameters()),
                  "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    require(parameters == config["parameters"], "E2 architecture/parameter counts changed")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device).eval()
    buffers_before = {name: value.detach().cpu().clone() for name, value in model.named_buffers()}
    loader = DataLoader(dataset, batch_size=32, num_workers=0, drop_last=False, shuffle=False)
    destination.mkdir(exist_ok=False)  # Never merge, resume or overwrite evaluation results.
    ids, true, predicted, confidence = [], [], [], []
    loss_sum = 0.0
    with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=False):
        for batch_number, (images, targets, image_ids) in enumerate(loader, 1):
            start, end = len(ids), len(ids) + len(image_ids)
            require(list(image_ids) == expected_ids[start:end] and targets.tolist() == expected_targets[start:end],
                    "Validation batch IDs/true labels differ from saved order")
            require(images.dtype == torch.float32 and targets.dtype == torch.int64
                    and tuple(images.shape) == (len(image_ids), 3, 224, 224), "Unexpected batch shape/dtype")
            images, targets = images.to(device), targets.to(device)
            require(bool(torch.isfinite(images).all()), "Nonfinite normalised inputs")
            logits = model(images)
            loss = torch.nn.functional.cross_entropy(logits, targets)
            require(logits.shape == (len(image_ids), 8) and logits.dtype == torch.float32
                    and bool(torch.isfinite(logits).all() & torch.isfinite(loss)), "Invalid logits or loss")
            maximum, indices = logits.softmax(dim=1).max(dim=1)
            loss_sum += float(loss) * len(image_ids)  # Sample weighting includes the final partial batch.
            ids.extend(image_ids)
            true.extend(targets.cpu().tolist())
            predicted.extend(indices.cpu().tolist())
            confidence.extend(maximum.cpu().tolist())
            if batch_number % 100 == 0 or batch_number == len(loader):
                print(f"E2 evaluation only: {len(ids):,}/12,733 validation images", flush=True)
    scores, per_class, matrix = _e2_validation_statistics(true, predicted, loss_sum, names)
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    checks = {
        "exact_unique_id_coverage_and_order": ids == expected_ids and len(set(ids)) == 12733,
        "true_labels_match": true == expected_targets,
        "class_supports_match": per_class["support"].tolist() == expected_support,
        "prediction_indices_in_range": all(0 <= value < 8 for value in predicted),
        "finite_metrics_and_confidences": bool(np.isfinite(list(scores.values())).all()
            and np.isfinite(per_class[["precision", "recall", "f1"]].to_numpy()).all() and np.isfinite(confidence).all()),
        "confidence_in_range": all(0 <= value <= 1 for value in confidence),
        "confusion_total_12733": int(matrix.sum()) == 12733,
        "confusion_supports_match": matrix.sum(axis=1).tolist() == expected_support,
        "normalised_rows_sum_to_one": bool(np.allclose(normalised.sum(axis=1), 1)),
        "eval_no_gradients": not model.training and all(p.grad is None for p in model.parameters()),
        "batchnorm_buffers_unchanged": all(torch.equal(value.cpu(), buffers_before[name]) for name, value in model.named_buffers()),
        "original_inputs_unchanged": all(digest(path) == before[name] for name, path in paths.items()),
    }
    require(all(checks.values()), "Evaluation integrity checks failed: " + str([name for name, passed in checks.items() if not passed]))
    comparison = {"epoch": 8, "history_accuracy": expected_accuracy, "history_loss": expected_loss,
        "accuracy_difference": scores["accuracy"] - expected_accuracy,
        "loss_difference": scores["cross_entropy"] - expected_loss,
        "accuracy_absolute_tolerance": 1e-12, "loss_absolute_tolerance": 1e-6, "relative_tolerance": 0,
        "accuracy_matches_within_tolerance": math.isclose(scores["accuracy"], expected_accuracy, rel_tol=0, abs_tol=1e-12),
        "loss_matches_within_tolerance": math.isclose(scores["cross_entropy"], expected_loss, rel_tol=0, abs_tol=1e-6)}
    discrepancy = not (comparison["accuracy_matches_within_tolerance"] and comparison["loss_matches_within_tolerance"])
    warnings = ["Restored metrics differ from epoch 8 outside stated tolerances; actual results retained."] if discrepancy else []
    predictions = pd.DataFrame({"image_id": ids, "true_class": [names[i] for i in true],
        "predicted_class": [names[i] for i in predicted], "max_softmax_confidence": confidence})
    with (destination / "validation_predictions.csv").open("x", newline="", encoding="utf-8") as stream:
        predictions.to_csv(stream, index=False)
    with (destination / "classification_report.csv").open("x", newline="", encoding="utf-8") as stream:
        per_class.to_csv(stream, index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), layout="constrained")
    for ax, values, title, fractional in zip(axes, [matrix, normalised],
            ["Raw counts", "Row-normalised (true class)"], [False, True]):
        image = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if fractional else matrix.max())
        ax.set(xticks=range(8), yticks=range(8), xticklabels=names, yticklabels=names,
               xlabel="Predicted class", ylabel="True class", title=title)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        for row in range(8):
            for column in range(8):
                value = values[row, column]
                ax.text(column, row, f"{value:.2f}" if fractional else str(value),
                        ha="center", va="center", fontsize=8.5,
                        color="white" if value > (0.5 if fractional else matrix.max() / 2) else "#25323A")
        fig.colorbar(image, ax=ax, shrink=0.8)
    fig.suptitle("E2 validation - saved checkpoint, epoch 8")
    with (destination / "confusion_matrix.png").open("xb") as stream:
        fig.savefig(stream, format="png", dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)
    files = [str(destination / name) for name in ["validation_metrics.json", "classification_report.csv",
             "validation_predictions.csv", "confusion_matrix.png"]]
    require(all(digest(path) == before[name] for name, path in paths.items()), "Original inputs changed during output saving")
    metrics = {"status": "passed_with_history_discrepancy" if discrepancy else "passed",
        "evaluation_only": True, "run_name": run.name, "best_epoch": 8, **scores,
        "device": str(device), "gpu": torch.cuda.get_device_name(0), "dtype": "float32", "amp": False,
        "batch_size": 32, "num_workers": 0, "drop_last": False, "shuffle": False, "augmentation": False,
        "parameters": parameters, "e1_checkpoint_provenance": config["e1_checkpoint"], "preprocessing": config["preprocessing"],
        "class_mapping": mapping, "per_class": per_class.to_dict(orient="records"),
        "confusion_matrix": matrix.tolist(), "row_normalised_confusion_matrix": normalised.tolist(),
        "input_sha256": before, "checks": checks, "failed_checks": [], "history_comparison": comparison,
        "warnings": warnings, "outputs": files,
        "runtime": {"python": platform.python_version(), "torch": str(torch.__version__),
                    "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__,
                    "Pillow": version("Pillow"), "cuda_build": torch.version.cuda},
        "limitations": ["No cross-version/GPU bitwise reproducibility claim.",
                        "Undefined precision/F1 uses zero_division=0; maximum softmax is not calibrated certainty."]}
    with (destination / "validation_metrics.json").open("x", encoding="utf-8") as stream:
        json.dump(metrics, stream, indent=2, allow_nan=False)
        stream.write("\n")
    print(json.dumps({"status": metrics["status"], **scores, "history_comparison": comparison,
                      "warnings": warnings, "outputs": files}, indent=2))
    return metrics


e2_validation_metrics = _evaluate_saved_e2_validation()


## 7. Guarded E2 test prediction - disabled by default

Copy **only the final code cell below** into the current Kaggle session after the successful E2 evaluation. It requires the existing `PartiallyFineTunedMobileNetV2`, `ImageNetNormalizedDataset`, `SETUP_ROOT`, `TRAIN_CACHE_DIR`, model `require` helper, and verified/imported `src.data_pipeline` definitions. Missing state/files stop clearly; no preparation, copying or training is restarted.

Keep `RUN_TEST_PREDICTION=False` until deliberately enabling this prediction cell. All training flags remain False. It requires E2 epoch-8 evaluation status passed without warnings, matching evaluated provenance and the pinned E2 checkpoint/configuration/history hashes. The model constructor uses weights=None, followed by strict restoration of the complete E2 state. No pretrained download occurs.

Discover sample_submission.csv and the unique complete test-image directory under `/kaggle/input/datasets/leocaoprojects/enge707-2026-s2/`. Preserve original string IDs and order; zfill(5) is used only to look up JPG filenames. Sample classes are placeholders and never become answers. Apply the unchanged reference geometry and E1/E2 ImageNet normalisation once, without augmentation. Check this against a validation-only example before inference. No test pre-decoding, image hashing, copying or dataset caching is performed.

**Input loading may dominate runtime: the previous E1 test prediction took approximately 39 minutes.** Print loading-inclusive timings for the first three batches (retaining their predictions), a clearly labelled rough projection, progress every 100 batches and actual total inference time. Do not assume cached-validation speed.

Write `/kaggle/working/submission_E2.csv` (exact image_id,class columns) and `/kaggle/working/submission_E2_check.json`. Check output conflicts before any inference and use exclusive creation. Verify the saved CSV's original IDs/order, valid classes and complete one-to-one filename coverage. Preserve original checkpoint/configuration/sample bytes and all E1 files. There is no automatic competition upload or hidden-test accuracy claim.


In [ ]:
# 7. GUARDED E2 TEST PREDICTION: enable deliberately; never upload automatically.
RUN_TEST_PREDICTION = False
RUN_E2_TRAINING = False
RUN_E1_TRAINING = False
RUN_FULL_TRAINING = False
import hashlib
import json
import math
import os
import platform
import re
import statistics
import sys
import time
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2


def _e2_discover_test_input(root):
    """Enumerate input filenames once; no image decoding or image-content hashing."""
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Supplied dataset directory unavailable: {root}")
    samples, image_directories = [], {}
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
    for directory, folders, files in os.walk(root, followlinks=False):
        directory = Path(directory)
        if any((directory / name).is_symlink() for name in folders):
            raise ValueError(f"Linked dataset directory is ambiguous: {directory}")
        for name in files:
            path = directory / name
            if name == "sample_submission.csv":
                if path.is_symlink():
                    raise ValueError("Sample submission must not be a link")
                samples.append(path)
            if "test" in directory.relative_to(root).parts and path.suffix.lower() in suffixes:
                if path.is_symlink():
                    raise ValueError(f"Linked test image is not allowed: {path}")
                image_directories.setdefault(directory, []).append(name)
    if len(samples) != 1 or len(image_directories) != 1:
        raise ValueError(f"Expected one sample CSV and one test-image directory; found {len(samples)} and {len(image_directories)}")
    directory, filenames = next(iter(image_directories.items()))
    return samples[0], directory.resolve(), filenames


def _e2_test_lookup(sample, filenames, expected_count=29443):
    if list(sample.columns) != ["image_id", "class"] or len(sample) != expected_count:
        raise ValueError("Expected sample columns image_id,class and exactly 29,443 rows")
    ids = sample["image_id"].tolist()
    if (any(not isinstance(i, str) or not re.fullmatch(r"[1-9][0-9]*", i) for i in ids)
            or len(set(ids)) != expected_count):
        raise ValueError("Expected unique nonempty original unpadded decimal string IDs")
    lookup = [image_id.zfill(5) + ".jpg" for image_id in ids]  # File lookup ONLY.
    if (len(set(lookup)) != expected_count or len(filenames) != expected_count
            or len(set(filenames)) != expected_count or set(lookup) != set(filenames)
            or any(not re.fullmatch(r"[0-9]{5}\.jpg", name) for name in filenames)):
        raise ValueError("Test filenames have missing, ambiguous, reused or unused images")
    return ids, lookup  # Never replace original sample IDs with padded IDs.


def _e2_test_tensor(image, preprocess, mean, std):
    tensor = preprocess(image)  # Existing RGB, bilinear resize/pad, float32 /255.
    if tensor.dtype != torch.float32 or tensor.shape != (3, 224, 224):
        raise ValueError("Unexpected reference-preprocessing output")
    return (tensor - mean) / std  # Same E2 normalisation, once; no crop/augmentation.


class E2TestDataset(Dataset):
    def __init__(self, directory, ids, lookup, preprocess, mean, std):
        self.directory = Path(directory).resolve()
        self.ids, self.lookup = list(ids), list(lookup)
        self.preprocess, self.mean, self.std = preprocess, mean, std

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        path = self.directory / self.lookup[index]
        if path.is_symlink() or path.resolve().parent != self.directory:
            raise ValueError(f"Test path escapes its directory: {path}")
        with Image.open(path) as image:
            image.load()  # Decode errors propagate; never skip an image.
            tensor = _e2_test_tensor(image, self.preprocess, self.mean, self.std)
        return tensor, self.ids[index]


def _e2_submission_frame(ids, indices, index_to_class, expected_count=29443):
    if (len(ids) != expected_count or len(set(ids)) != expected_count or len(indices) != expected_count
            or any(not isinstance(i, str) or not i.strip() for i in ids)
            or any(not isinstance(i, (int, np.integer)) or not 0 <= i < 8 for i in indices)):
        raise ValueError("Incomplete IDs or invalid prediction indices")
    classes = [index_to_class[str(index)] for index in indices]
    if not set(classes) <= set(index_to_class.values()):
        raise ValueError("Prediction outside official class encoding")
    return pd.DataFrame({"image_id": list(ids), "class": classes})


def _predict_e2_test():
    working = Path("/kaggle/working")
    submission_path, check_path = working / "submission_E2.csv", working / "submission_E2_check.json"
    for path in [submission_path, check_path]:
        if path.exists() or path.is_symlink():
            raise FileExistsError(f"Existing output left unchanged: {path}")
    missing = [name for name in ["PartiallyFineTunedMobileNetV2", "ImageNetNormalizedDataset", "SETUP_ROOT", "TRAIN_CACHE_DIR", "require"] if name not in globals()]
    if missing:
        raise RuntimeError("Missing verified session definitions: " + ", ".join(missing) + "; no setup rerun")
    run = working / "E2_mobilenetv2_finetuned_seed42"
    setup = Path(SETUP_ROOT).resolve()
    evaluation_path = run / "evaluation/validation_metrics.json"
    if not evaluation_path.is_file():
        raise FileNotFoundError(f"Successful E2 evaluation is required: {evaluation_path}")
    evaluation = json.loads(evaluation_path.read_text())
    if (evaluation.get("status") != "passed" or evaluation.get("samples") != 12733
            or evaluation.get("run_name") != run.name or evaluation.get("best_epoch") != 8
            or evaluation.get("evaluation_only") is not True or evaluation.get("warnings")
            or evaluation.get("failed_checks") or not evaluation.get("checks")
            or not all(v is True for v in evaluation["checks"].values())):
        raise ValueError("E2 evaluation did not pass all checks without warnings")
    expected_scores = {"accuracy": 0.9862561847168774, "cross_entropy": 0.040039216507934836,
                       "macro_f1": 0.9873287581357201}
    score_tolerances = {"accuracy": 1e-12, "cross_entropy": 1e-6, "macro_f1": 1e-12}
    if any(not math.isclose(evaluation[key], value, rel_tol=0, abs_tol=score_tolerances[key])
           for key, value in expected_scores.items()):
        raise ValueError("Evaluation differs from the verified E2 results")
    comparison = evaluation["history_comparison"]
    if (comparison.get("epoch") != 8
            or not comparison.get("accuracy_matches_within_tolerance")
            or not comparison.get("loss_matches_within_tolerance")
            or not math.isclose(comparison["accuracy_difference"], 0, rel_tol=0, abs_tol=1e-12)
            or not math.isclose(comparison["loss_difference"], 0, rel_tol=0, abs_tol=1e-6)):
        raise ValueError("Successful epoch-8 history comparison is required")

    def digest(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    pinned = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
    if digest(setup / "transfer_manifest.json") != pinned:
        raise ValueError("Pinned setup manifest changed")
    transfer = json.loads((setup / "transfer_manifest.json").read_text())
    payload_names = {"src/__init__.py", "src/data_pipeline.py", "src/models.py",
                     "data/processed/split_manifest.csv", "data/processed/class_mapping.json",
                     "preprocessing_fingerprints.json"}
    if set(transfer["sha256"]) != payload_names:
        raise ValueError("Unexpected setup payload")
    expected_hashes = {**transfer["sha256"], "transfer_manifest": pinned}
    inputs = {"checkpoint": run / "best_checkpoint.pt", "configuration": run / "configuration.json",
              "history": run / "history.csv", "completion_status": run / "completion_status.json",
              "cache_report": working / "training_image_cache_check.json",
              "transfer_manifest": setup / "transfer_manifest.json"}
    inputs.update({name: setup / name for name in payload_names})
    for name, path in inputs.items():
        if not path.is_file() or path.is_symlink():
            raise FileNotFoundError(f"Required existing E2/setup file unavailable or unsafe: {path}; nothing recreated")
        if name in payload_names and not path.resolve().is_relative_to(setup):
            raise ValueError("Setup payload escapes the verified setup directory: " + name)
    before = {name: digest(path) for name, path in inputs.items()}
    pinned_e2 = {"checkpoint": "89718e38fc47bff1fa31ed923ebab9a5a2aefe7fdea27fe73334cb366443d688",
                 "configuration": "98c31a5c619f3c9a44731cf96bb43ea75955c071f48345b54a862658cbaf2c90",
                 "history": "ea2f434868afc1edc86205f41065099635924bb06cb8c52b5b26833eeeca84b4"}
    if (before != evaluation["input_sha256"]
            or any(before[name] != value for name, value in expected_hashes.items())
            or any(before[name] != value for name, value in pinned_e2.items())):
        raise ValueError("Evaluated E2 checkpoint/configuration/source/split/mapping hashes changed")
    inputs["evaluation"] = evaluation_path
    before["evaluation"] = digest(evaluation_path)
    config = json.loads(inputs["configuration"].read_text())
    mapping = json.loads((setup / "data/processed/class_mapping.json").read_text())
    names = [mapping["index_to_class"][str(i)] for i in range(8)]
    checkpoint = torch.load(inputs["checkpoint"], weights_only=True, map_location="cpu")
    if (config["run_name"] != run.name or checkpoint["configuration"] != config or checkpoint["best_epoch"] != 8
            or checkpoint["class_mapping"] != mapping or config["class_mapping"] != mapping
            or evaluation["class_mapping"] != mapping or transfer["classes"] != mapping
            or names != sorted(names) or len(set(names)) != 8
            or mapping["class_to_index"] != dict(zip(names, range(8)))
            or checkpoint["split_checksum"] != before["data/processed/split_manifest.csv"]
            or config["split_sha256"] != checkpoint["split_checksum"]
            or checkpoint["best_validation_loss"] != expected_scores["cross_entropy"]
            or config["preprocessing"] != evaluation["preprocessing"]):
        raise ValueError("E2 checkpoint/configuration/class encoding is inconsistent")
    history = pd.read_csv(inputs["history"], float_precision="round_trip")
    completion = json.loads(inputs["completion_status"].read_text())
    if (history["epoch"].tolist() != list(range(1, 9))
            or not history["training_samples"].eq(50927).all()
            or not history["validation_samples"].eq(12733).all()
            or int(history.loc[history["validation_loss"].idxmin(), "epoch"]) != 8
            or float(history.iloc[-1]["validation_accuracy"]) != expected_scores["accuracy"]
            or float(history.iloc[-1]["validation_loss"]) != expected_scores["cross_entropy"]
            or completion["status"] != "completed" or completion["completed_epochs"] != 8
            or completion["best_epoch"] != 8 or completion["best_validation_loss"] != expected_scores["cross_entropy"]
            or completion["failures"] or not completion["frozen_parameters_and_bn_buffers_unchanged"]
            or completion["history_sha256"] != before["history"]):
        raise ValueError("Successful completed E2 epoch-8 history/metadata required")
    module = sys.modules.get("src.data_pipeline")
    if module is None or not getattr(module, "__file__", None) or Path(module.__file__).resolve() != setup / "src/data_pipeline.py":
        raise RuntimeError("Verified preprocessing/Dataset module is unavailable")
    reference_preprocess, FacialFeatureDataset = module.reference_preprocess, module.FacialFeatureDataset
    data_root = Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2/")
    sample_path, test_directory, filenames = _e2_discover_test_input(data_root)
    inputs["sample_submission"] = sample_path
    before["sample_submission"] = digest(sample_path)
    sample = pd.read_csv(sample_path, dtype=str, keep_default_na=False, na_filter=False)
    original_ids, lookup = _e2_test_lookup(sample, filenames)
    # Sample 'class' entries are placeholders and are never consulted.
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]
    cache = Path(TRAIN_CACHE_DIR).resolve()
    if cache != Path(config["training_image_directory"]).resolve():
        raise ValueError("Existing validation-image cache differs from E2 configuration")
    cache_report = json.loads(inputs["cache_report"].read_text())
    if (not cache.is_dir() or cache_report["status"] != "passed" or cache_report.get("failures")
            or cache_report.get("hash_mismatches") or cache_report.get("verified_files") != 63660
            or Path(cache_report["cache_directory"]).resolve() != cache):
        raise RuntimeError("Existing validation-image cache needed for the preprocessing check is unavailable")
    validation_base = FacialFeatureDataset(setup, "validation", training_image_directory=cache)
    validation = ImageNetNormalizedDataset(validation_base)
    if (validation_base.augmentation is not None or not torch.equal(validation.mean, mean)
            or not torch.equal(validation.std, std) or config["preprocessing"]["normalisation_applications"] != 1):
        raise ValueError("E2 validation normalisation/augmentation changed")
    expected_tensor, _, validation_id = validation[0]
    with Image.open(cache / (validation_id + ".jpg")) as image:
        image.load()
        test_tensor = _e2_test_tensor(image, reference_preprocess, mean, std)
    if not torch.equal(test_tensor, expected_tensor) or not bool(torch.isfinite(test_tensor).all()):
        raise ValueError("Test preprocessing differs from E2 validation preprocessing")
    if not torch.cuda.is_available():
        raise RuntimeError("Required cuda:0 is unavailable")
    device = torch.device("cuda:0")
    model = PartiallyFineTunedMobileNetV2().to(dtype=torch.float32)  # Constructor uses weights=None; no download.
    parameters = {"total": sum(p.numel() for p in model.parameters()),
                  "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    if parameters != config["parameters"] or parameters != {"total": 2234120, "trainable": 1205128}:
        raise ValueError("E2 model architecture changed")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device).eval()
    dataset = E2TestDataset(test_directory, original_ids, lookup, reference_preprocess, mean, std)
    loader = DataLoader(dataset, batch_size=32, num_workers=0, drop_last=False, shuffle=False)
    predicted_ids, predicted_indices, first_batch_seconds = [], [], []
    rough_projection = None
    print("Input loading may dominate runtime: the previous E1 test prediction took approximately 39 minutes.", flush=True)
    torch.cuda.synchronize(device)
    started = time.perf_counter()
    iterator = iter(loader)
    with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=False):
        for batch_number in range(1, len(loader) + 1):
            torch.cuda.synchronize(device)
            batch_started = time.perf_counter()
            images, ids = next(iterator)  # First batches are retained, never rerun.
            start, end = len(predicted_ids), len(predicted_ids) + len(ids)
            if list(ids) != original_ids[start:end] or images.dtype != torch.float32 or images.shape != (len(ids), 3, 224, 224):
                raise ValueError("Test batch order/shape/precision changed")
            images = images.to(device)
            if not bool(torch.isfinite(images).all()):
                raise FloatingPointError("Nonfinite test tensors")
            logits = model(images)
            if logits.shape != (len(ids), 8) or logits.dtype != torch.float32 or not bool(torch.isfinite(logits).all()):
                raise FloatingPointError("Invalid/nonfinite test logits")
            predicted_indices.extend(logits.argmax(dim=1).cpu().tolist())
            predicted_ids.extend(ids)
            torch.cuda.synchronize(device)
            duration = time.perf_counter() - batch_started
            if batch_number <= 3:
                first_batch_seconds.append(duration)
                print(f"Test batch {batch_number}: {duration:.3f}s (input loading included; predictions retained)", flush=True)
            if batch_number == min(3, len(loader)):
                rough_projection = {"label": "Rough projection from first input-mount batches, NOT measured full runtime",
                    "sample_batches": len(first_batch_seconds), "seconds": statistics.mean(first_batch_seconds) * len(loader)}
                print(json.dumps(rough_projection), flush=True)
            if batch_number % 100 == 0 or batch_number == len(loader):
                print(f"Test inference: {len(predicted_ids):,}/{len(original_ids):,} images", flush=True)
    torch.cuda.synchronize(device)
    inference_seconds = time.perf_counter() - started
    submission = _e2_submission_frame(predicted_ids, predicted_indices, mapping["index_to_class"])
    checks = {"successful_e2_evaluation": True, "complete_one_to_one_filename_coverage": True,
        "preprocessing_matches_validation": True, "finite_tensors_and_eight_logits": True,
        "exact_original_id_order": predicted_ids == original_ids,
        "unique_complete_29443_predictions": len(submission) == 29443 and submission["image_id"].is_unique,
        "official_classes_only": bool(submission["class"].isin(names).all()),
        "original_inputs_unchanged": all(digest(path) == before[name] for name, path in inputs.items())}
    if not all(checks.values()):
        raise ValueError("Submission checks failed: " + str(checks))
    with submission_path.open("x", newline="", encoding="utf-8") as stream:
        submission.to_csv(stream, index=False)  # Exact image_id,class columns; no index.
    reread = pd.read_csv(submission_path, dtype=str, keep_default_na=False, na_filter=False)
    checks["saved_csv_roundtrip"] = list(reread.columns) == ["image_id", "class"] and reread.to_dict("list") == submission.to_dict("list")
    saved_ids, saved_lookup = _e2_test_lookup(reread, filenames)
    checks["saved_csv_complete_image_coverage"] = saved_ids == original_ids and saved_lookup == lookup
    checks["saved_csv_unique_ids_and_official_classes"] = (len(reread) == 29443
        and reread["image_id"].is_unique and reread["class"].isin(names).all())
    checks["original_inputs_unchanged"] = all(digest(path) == before[name] for name, path in inputs.items())
    # pandas/NumPy reductions can return np.bool_; JSON requires native bool values.
    checks = {name: bool(passed) for name, passed in checks.items()}
    if not all(checks.values()):
        raise ValueError("Saved submission failed string-ID/content round-trip; file retained for inspection")
    result = {"status": "passed", "run_name": run.name, "best_epoch": 8, "checks": checks, "failed_checks": [], "row_count": len(submission),
        "class_counts": {name: int(submission["class"].eq(name).sum()) for name in names},
        "checkpoint_sha256": before["checkpoint"], "configuration_sha256": before["configuration"],
        "sample_sha256": before["sample_submission"], "submission_sha256": digest(submission_path),
        "input_sha256": before, "test_directory": str(test_directory),
        "checkpoint_provenance": {"best_epoch": 8, "best_validation_loss": checkpoint["best_validation_loss"],
                                  "e1_origin": config["e1_checkpoint"]},
        "evaluation_scores": {key: evaluation[key] for key in expected_scores},
        "id_mapping": "Original sample ID preserved; zfill(5) + '.jpg' for lookup only",
        "preprocessing": config["preprocessing"], "parameters": parameters,
        "device": str(device), "gpu": torch.cuda.get_device_name(0), "dtype": "float32", "amp": False,
        "batch_size": 32, "num_workers": 0, "drop_last": False, "shuffle": False,
        "first_batch_seconds_loading_inclusive": first_batch_seconds, "rough_runtime_projection": rough_projection,
        "inference_seconds_loading_inclusive": inference_seconds, "completed_batches": len(loader),
        "runtime": {"python": platform.python_version(), "torch": str(torch.__version__), "torchvision": torchvision.__version__,
                    "numpy": np.__version__, "pandas": pd.__version__, "Pillow": version("Pillow"), "cuda_build": torch.version.cuda},
        "outputs": [str(submission_path), str(check_path)],
        "limitations": ["Predicted class counts are not hidden-test labels or accuracy.",
                        "Input-mount timings include startup/cache effects; no cached-validation speed assumption.",
                        "No competition upload was performed."]}
    # Serialize the complete report before opening its exclusive output file.
    report_json = json.dumps(result, indent=2, allow_nan=False) + "\n"
    with check_path.open("x", encoding="utf-8") as stream:
        stream.write(report_json)
    print(json.dumps({key: result[key] for key in ["status", "row_count", "class_counts", "inference_seconds_loading_inclusive", "outputs"]}, indent=2))
    return result


if RUN_TEST_PREDICTION:
    submission_E2_check = _predict_e2_test()
else:
    print("RUN_TEST_PREDICTION=False: no test images decoded, predictions or submission files created.")